# DrishtiYOLO — Gradio Demo Notebook
Run this in Colab to get a public demo URL, or locally after training.

In [ ]:
!pip install -q ultralytics gradio

from google.colab import drive
drive.mount('/content/drive')

BEST_PT = '/content/drive/MyDrive/drishti_runs/best_yolo11s_idd_v1_final.pt'


In [ ]:
import gradio as gr
import numpy as np
from PIL import Image
from ultralytics import YOLO

model = YOLO(BEST_PT)

CLASSES = [
    'auto_rickshaw', 'bicycle', 'bus', 'car', 'caravan',
    'electric_vehicle', 'motorcycle', 'person', 'rider',
    'traffic_light', 'traffic_sign', 'truck',
    'vehicle_fallback', 'animal', 'wheelchair'
]

def predict(image: np.ndarray, conf_thresh: float):
    results = model(image, conf=conf_thresh, verbose=False)
    annotated = results[0].plot()
    # Build detection summary
    boxes = results[0].boxes
    if boxes is None or len(boxes) == 0:
        summary = 'No detections above threshold.'
    else:
        counts = {}
        for cls_id in boxes.cls.cpu().numpy().astype(int):
            name = CLASSES[cls_id] if cls_id < len(CLASSES) else str(cls_id)
            counts[name] = counts.get(name, 0) + 1
        summary = '  ·  '.join(f'{v}× {k}' for k, v in sorted(counts.items(), key=lambda x: -x[1]))
    return Image.fromarray(annotated), summary


with gr.Blocks(title='DrishtiYOLO') as demo:
    gr.Markdown(
        '# DrishtiYOLO\n'
        'YOLO11s fine-tuned on the Indian Driving Dataset (IDD).  '
        'Detects India-specific objects — auto-rickshaws, riders, cattle — '
        'that off-the-shelf COCO models miss.'
    )
    with gr.Row():
        with gr.Column():
            inp_img  = gr.Image(label='Input image', type='numpy')
            conf_sl  = gr.Slider(0.1, 0.9, value=0.25, step=0.05, label='Confidence threshold')
            run_btn  = gr.Button('Detect', variant='primary')
        with gr.Column():
            out_img  = gr.Image(label='Annotated output')
            out_text = gr.Textbox(label='Detections', lines=2)

    gr.Examples(
        examples=[],   # add local example image paths here
        inputs=inp_img,
    )

    run_btn.click(predict, inputs=[inp_img, conf_sl], outputs=[out_img, out_text])

demo.launch(share=True)  # share=True gives a public link from Colab
